# 03. Model Design — Advanced (M1): Random Forest

**Judul proyek:** Deteksi Dini Risiko Kecemasan Sedang–Berat pada Gamer Online

**Fokus lifecycle:** Bab 6.2.1–6.2.3 — Advanced Model Design, Training, Validation, dan Comparison

Notebook ini memakai split dan definisi target yang sama dengan notebook baseline agar perbandingan tidak bias. Angka yang dilaporkan adalah hasil **validation set**, sedangkan hold-out test belum digunakan.

## 6.2.1 Pemilihan model advanced: Random Forest

Model advanced yang dipilih adalah **Random Forest Classifier**, yaitu ensemble *bagging* dari banyak decision tree. Setiap pohon dilatih pada bootstrap sample dan hanya melihat subset fitur acak saat mencari split; probabilitas akhir adalah rerata probabilitas semua pohon.

**Mengapa sesuai dengan data ini:** hubungan antara jam bermain, faktor psikologis, demografi, dan risiko kecemasan kemungkinan mengandung ambang serta interaksi non-linear. Random Forest dapat menangkap pola tersebut tanpa perlu menspesifikasikan bentuk interaksi secara manual, tahan terhadap skala fitur, dan tetap memberi *feature importance*.

### Keunggulan, kelemahan, hypothesis space, dan inductive bias

**Keunggulan:** mampu memodelkan non-linearitas/interaksi, lebih stabil daripada satu decision tree, relatif tahan outlier, dan bekerja baik pada fitur campuran setelah encoding.

**Kelemahan:** lebih sulit dijelaskan per individu dibanding Logistic Regression, lebih mahal secara komputasi, *impurity importance* dapat bias pada fitur berkardinalitas tinggi, dan probabilitas kadang perlu kalibrasi.

- **Hypothesis space:** fungsi *piecewise-constant* hasil rata-rata banyak pohon keputusan; tiap pohon membagi ruang fitur secara rekursif menggunakan split sumbu-sejajar.
- **Inductive bias:** responden dalam wilayah fitur yang sama dianggap serupa; banyak split sederhana dapat membentuk pola kompleks; bagging dan pemilihan fitur acak mengurangi varians serta korelasi antarpohon. `min_samples_leaf=2` membatasi daun yang terlalu spesifik.

## 6.2.2 Import, pemuatan data, dan split identik

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

data_path = Path('../data/GamingStudy_data.csv')
if not data_path.exists():
    data_path = Path('data/GamingStudy_data.csv')
df = pd.read_csv(data_path, encoding='latin1')
df = df[(df['accept'] == 'Accept') & df['Hours'].notna() & (df['Hours'] <= 84)].copy()
y = (df['GAD_T'] >= 10).astype(int)

leakage_cols = ['GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD_T', 'GADE']
uninformative_cols = ['S. No.', 'Timestamp', 'accept', 'highestleague', 'League',
                      'Birthplace', 'Residence', 'Reference', 'Residence_ISO3', 'Birthplace_ISO3']
X = df.drop(columns=leakage_cols + uninformative_cols).copy()

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=RANDOM_SEED
)
print(f'Train: {len(X_train):,} | Validation: {len(X_val):,} | Hold-out test: {len(X_test):,}')
print(f'Proporsi positif validation: {y_val.mean():.2%}')

Train: 7,788 | Validation: 2,596 | Hold-out test: 2,597
Proporsi positif validation: 17.37%


## 6.2.3 Preprocessing bebas leakage

Transformasi sama dengan baseline. Fitting imputasi, scaling, dan encoder terjadi di dalam `Pipeline.fit(X_train, y_train)`, sehingga Random Forest tidak pernah melihat informasi dari validation/test ketika mempelajari transformasi.

In [2]:
top_games = ['League of Legends', 'Counter Strike', 'Starcraft 2', 'World of Warcraft', 'Other']
top_styles = ['Multiplayer - online - with real life friends',
              'Multiplayer - online - with strangers',
              'Multiplayer - online - with online acquaintances or teammates', 'Singleplayer']
top_why = ['having fun', 'improving', 'winning', 'relaxing']

for frame in (X_train, X_val, X_test):
    frame['Game_Clean'] = frame['Game'].where(frame['Game'].isin(top_games), 'Other')
    frame['Playstyle_Clean'] = frame['Playstyle'].where(frame['Playstyle'].isin(top_styles), 'Other')
    frame['whyplay_Clean'] = frame['whyplay'].where(frame['whyplay'].isin(top_why), 'Other')
    frame['SPIN_SWL_Ratio'] = frame['SPIN_T'] / (frame['SWL_T'] + 1.0)
    frame.drop(columns=['Game', 'Playstyle', 'whyplay'], inplace=True)

numeric_cols = ['Hours', 'streams', 'Age', 'SWL_T', 'SPIN_T', 'Narcissism', 'SPIN_SWL_Ratio']
categorical_cols = ['Platform', 'Gender', 'Work', 'Degree', 'Game_Clean', 'Playstyle_Clean', 'whyplay_Clean']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler', RobustScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
])

### Hyperparameter awal

| Hyperparameter | Nilai | Justifikasi |
|---|---:|---|
| `n_estimators` | 400 | Mengurangi varians prediksi dibanding sedikit pohon |
| `max_features` | `sqrt` | Mendekorelasikan pohon dan lazim untuk klasifikasi |
| `min_samples_leaf` | 2 | Menahan split yang terlalu spesifik |
| `class_weight` | `balanced_subsample` | Menyeimbangkan biaya kelas pada setiap bootstrap sample |
| `max_depth` | `None` | Membolehkan interaksi kompleks; regulasi dilakukan oleh leaf minimum dan ensemble |
| `random_state` | 42 | Hasil dapat direproduksi |
| `n_jobs` | -1 | Memakai core yang tersedia saat fitting |

In [3]:
advanced = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=400, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced_subsample', max_depth=None,
        random_state=RANDOM_SEED, n_jobs=-1
    ))
])
advanced.fit(X_train, y_train)
print(f'Random Forest berhasil dilatih pada {len(X_train):,} sampel train.')

Random Forest berhasil dilatih pada 7,788 sampel train.


## Hasil Random Forest pada validation set

In [4]:
val_pred = advanced.predict(X_val)
val_proba = advanced.predict_proba(X_val)[:, 1]
metrics_advanced = pd.DataFrame({
    'metric': ['Accuracy', 'Precision (positive)', 'Recall (positive)', 'F1 (positive)', 'Macro-F1', 'ROC-AUC', 'PR-AUC'],
    'score': [
        accuracy_score(y_val, val_pred),
        precision_score(y_val, val_pred, zero_division=0),
        recall_score(y_val, val_pred),
        f1_score(y_val, val_pred, zero_division=0),
        f1_score(y_val, val_pred, average='macro'),
        roc_auc_score(y_val, val_proba),
        average_precision_score(y_val, val_proba),
    ]
})
print(metrics_advanced.to_string(index=False))
print('\nConfusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_val, val_pred))

              metric     score
            Accuracy  0.833975
Precision (positive)  0.533113
   Recall (positive)  0.356984
    F1 (positive)     0.427623
            Macro-F1  0.665264
             ROC-AUC  0.773437
              PR-AUC  0.461923

Confusion matrix [[TN, FP], [FN, TP]]:
[[2004  141]
 [ 290  161]]


## 6.2.3 Perbandingan awal: baseline vs advanced

| Metrik validation | Logistic Regression (baseline) | Random Forest (advanced) |
|---|---:|---:|
| Accuracy | 0,721 | **0,834** |
| Precision positif | 0,347 | **0,533** |
| Recall positif | **0,690** | 0,357 |
| F1 positif | **0,462** | 0,428 |
| Macro-F1 | 0,637 | **0,665** |
| ROC-AUC | **0,783** | 0,773 |
| PR-AUC | **0,489** | 0,462 |

Random Forest lebih presisi dan mempunyai Macro-F1/akurasi lebih tinggi, tetapi Logistic Regression jauh lebih sensitif mendeteksi responden berisiko. Untuk konteks skrining dini, Recall positif menjadi sangat penting; oleh karena itu baseline belum boleh dieliminasi hanya berdasarkan akurasi. Tahap berikutnya yang tepat adalah memilih objective (misalnya minimum recall), menala hyperparameter/threshold **hanya pada validation set**, kemudian melakukan evaluasi final satu kali pada hold-out test.

### Kesimpulan checkpoint model design

- Kedua model memakai fitur, preprocessing, seed, dan validation set yang identik.
- Tidak ada fitur GAD yang dipakai sebagai masukan, sehingga label leakage terhindari.
- Hasil yang tercatat seluruhnya berasal dari validation set; hold-out test tidak disentuh.
- Ada trade-off nyata antara sensitivity baseline dan precision/overall balance Random Forest. Keputusan model akhir harus mengikuti biaya false negative pada konteks deteksi dini.